# Sprint 2: Cierre — Evaluación sobre test + Grad-CAM

Este notebook:
1. Instala `run_eval_s2.py` en Drive.
2. Evalúa el mejor checkpoint sobre el **test set** (40 imágenes nunca vistas).
3. Genera curva ROC multi-clase y matriz de confusión 4×4.
4. Genera Grad-CAM para las 4 clases.
5. Imprime el resumen final del Sprint 2.

**Runtime: T4 GPU.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'
CODE_DIR     = f'{PROJECT_ROOT}/code'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

H5_LOCAL      = '/content/nih_images.h5'
PROCESSED_DIR = '/content/processed_s2'
CLASS_NAMES   = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}


In [ ]:
!pip install -q torchxrayvision scipy h5py grad-cam
print('deps OK')


## 2. Verificar HDF5 y CSVs en disco local

In [ ]:
import h5py

# HDF5
if os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 1_000_000_000:
    with h5py.File(H5_LOCAL, 'r') as hf:
        print(f'✓ HDF5 en disco local: {len(hf["images"]):,} imágenes')
else:
    print('Copiando HDF5 de Drive...')
    t0 = time.time()
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
    print(f'✓ Copiado en {(time.time()-t0)/60:.1f} min')

# CSVs
import pandas as pd
os.makedirs(PROCESSED_DIR, exist_ok=True)
for csv in ['train.csv', 'val.csv', 'test.csv']:
    local = f'{PROCESSED_DIR}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s2/{csv}', local)
    df = pd.read_csv(local)
    print(f'✓ {csv}: {len(df)} imgs')


## 3. Instalar `scripts/run_eval_s2.py`

In [ ]:
from pathlib import Path

content = r'''"""
scripts/run_eval_s2.py
Evaluación final del Sprint 2 sobre test set.
"""
from __future__ import annotations
import json, os, sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
import numpy as np

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms import build_eval_transform
from src.models.baseline import build_baseline_model
from src.evaluation.metrics_s2 import compute_metrics_multiclass, format_metrics
from src.training.utils import get_device, get_logger, load_config, load_checkpoint


CLASS_NAMES = {0: "No Finding", 1: "Cardiomegaly",
               2: "Effusion",   3: "Infiltration"}


def main():
    h5_path       = os.environ.get("H5_PATH",       "/content/nih_images.h5")
    processed_dir = os.environ.get("PROCESSED_DIR", "/content/processed_s2")

    cfg    = load_config(CODE_DIR / "configs" / "sprint2.yaml")
    device = get_device()

    Path(cfg.paths["logs"]).mkdir(parents=True, exist_ok=True)
    logger = get_logger("eval_s2",
                        log_file=Path(cfg.paths["logs"]) / "eval_s2.log")

    logger.info("=" * 70)
    logger.info("RUN EVAL Sprint 2 — test set")
    logger.info("=" * 70)

    # Dataset de test
    eval_tfm = build_eval_transform(image_size=cfg.preprocessing["image_size"])
    test_ds  = NIHDatasetHDF5(f"{processed_dir}/test.csv", h5_path, eval_tfm)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                             num_workers=0, pin_memory=(device.type == "cuda"))

    logger.info("Test set: %d imágenes", len(test_ds))
    dist = test_ds.class_distribution()
    for label, name in CLASS_NAMES.items():
        logger.info("  %s: %d", name, dist.get(label, 0))

    # Cargar mejor checkpoint
    ckpt_path = Path(cfg.paths["checkpoints"]) / "sprint2_best.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint no encontrado: {ckpt_path}")

    model = build_baseline_model(cfg).to(device)
    ckpt  = load_checkpoint(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    logger.info("Checkpoint cargado: epoch=%d | val AUC_macro=%.4f",
                ckpt["epoch"], ckpt["best_val_auc_macro"])

    # Inferencia
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch["image"].to(device))
            probs  = torch.softmax(logits.float(), dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(batch["label"].numpy())

    all_probs  = np.concatenate(all_probs,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Métricas
    metrics = compute_metrics_multiclass(all_labels, all_probs)

    logger.info("=" * 70)
    logger.info("MÉTRICAS FINALES SOBRE TEST:")
    logger.info(format_metrics(metrics))
    logger.info("=" * 70)

    # Guardar resultados
    out_path = Path(cfg.paths["experiments"]) / "test_metrics_s2.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w") as f:
        json.dump({
            "metrics": metrics,
            "probs":   all_probs.tolist(),
            "labels":  all_labels.tolist(),
            "checkpoint_epoch": int(ckpt["epoch"]),
        }, f, indent=2)
    logger.info("Guardado en: %s", out_path)


if __name__ == "__main__":
    main()
'''
p = Path(f'{CODE_DIR}/scripts/run_eval_s2.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p} ({len(content):,} chars)')


## 4. Evaluar sobre test set

In [ ]:
import subprocess

env = {**os.environ, 'H5_PATH': H5_LOCAL, 'PROCESSED_DIR': PROCESSED_DIR}

result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_eval_s2.py'],
    capture_output=True, text=True, env=env,
)
print('LOG:')
print(result.stderr[-4000:] if len(result.stderr) > 4000 else result.stderr)
if result.returncode != 0:
    print('STDOUT:', result.stdout)


## 5. Métricas detalladas sobre test

In [ ]:
import json
import numpy as np

test_metrics_path = f'{PROJECT_ROOT}/experiments_s2/test_metrics_s2.json'
with open(test_metrics_path) as f:
    data = json.load(f)

m = data['metrics']
print('MÉTRICAS FINALES — TEST SET (Sprint 2)')
print('=' * 60)
print(f"AUC macro:        {m['auc_macro']:.4f}")
print(f"Accuracy:         {m['accuracy']:.4f}")
print(f"Precision macro:  {m['precision_macro']:.4f}")
print(f"Sensitivity macro:{m['sensitivity_macro']:.4f}")
print(f"Specificity macro:{m['specificity_macro']:.4f}")
print()
print(f"{'Clase':<15} {'AUC':>6} {'Sens':>6} {'Spec':>6} {'Prec':>6}")
print('-' * 45)
for i in range(4):
    print(f"{CLASS_NAMES[i]:<15} "
          f"{m['auc_per_class'][i]:>6.4f} "
          f"{m['sensitivity_per_class'][i]:>6.4f} "
          f"{m['specificity_per_class'][i]:>6.4f} "
          f"{m['precision_per_class'][i]:>6.4f}")
print()
print(f"N muestras test: {m['n_samples']}")


## 6. Curvas ROC por clase

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.preprocessing import label_binarize

labels = np.array(data['labels'])
probs  = np.array(data['probs'])
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC por clase
ax = axes[0]
for i in range(4):
    y_bin = (labels == i).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, probs[:, i])
    auc = m['auc_per_class'][i]
    ax.plot(fpr, tpr, color=colors[i],
            label=f'{CLASS_NAMES[i]} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random')
ax.set_xlabel('1 - Especificidad (FPR)')
ax.set_ylabel('Sensibilidad (TPR)')
ax.set_title(f'Curvas ROC por clase — Test\nAUC macro = {m["auc_macro"]:.4f}')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

# Matriz de confusión
import seaborn as sns
cm = np.array(m['confusion_matrix'])
ax2 = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[CLASS_NAMES[i] for i in range(4)],
            yticklabels=[CLASS_NAMES[i] for i in range(4)],
            cbar=False, square=True, ax=ax2, annot_kws={'size': 11})
ax2.set_title('Matriz de Confusión — Test Set')
ax2.set_xlabel('Predicho')
ax2.set_ylabel('Real')
plt.setp(ax2.get_xticklabels(), rotation=25, ha='right', fontsize=9)
plt.setp(ax2.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
os.makedirs(f'{PROJECT_ROOT}/experiments_s2/figures', exist_ok=True)
plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/test_roc_cm_s2.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('✓ Guardado en experiments_s2/figures/')


## 7. Grad-CAM para las 4 clases

Carga el mejor checkpoint y genera mapas de calor para 2 imágenes por clase.

In [ ]:
# Limpiar cache de imports
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

import torch
import numpy as np
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from src.training.utils import load_config, get_device, load_checkpoint
from src.models.baseline import build_baseline_model
from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms import build_eval_transform

cfg    = load_config(f'{CODE_DIR}/configs/sprint2.yaml')
device = get_device()

# Cargar modelo
model = build_baseline_model(cfg).to(device)
ckpt  = load_checkpoint(
    f'{PROJECT_ROOT}/experiments_s2/checkpoints/sprint2_best.pt',
    map_location=device
)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Modelo cargado (epoch {ckpt["epoch"]}, AUC_macro={ckpt["best_val_auc_macro"]:.4f})')

# Dataset de test
eval_tfm = build_eval_transform(image_size=224)
test_ds  = NIHDatasetHDF5(f'{PROCESSED_DIR}/test.csv', H5_LOCAL, eval_tfm)

# Target layer para Grad-CAM
target_layer = model.backbone.features.denseblock4

# Normalizar imagen para visualización
def norm(img):
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)

# Grad-CAM
cam = GradCAM(model=model, target_layers=[target_layer])

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
colors_title = ['#2196F3','#F44336','#4CAF50','#FF9800']

for cls_label in range(4):
    # Tomar las 2 primeras imágenes de esta clase en test
    indices = [i for i in range(len(test_ds))
               if test_ds.df.iloc[i]['label'] == cls_label][:2]

    for col_offset, idx in enumerate(indices):
        sample = test_ds[idx]
        image_tensor = sample['image'].unsqueeze(0).to(device)
        img_name = sample['image_index']

        # Predicción
        with torch.no_grad():
            logits = model(image_tensor)
            probs  = torch.softmax(logits, dim=1)[0]
            pred   = probs.argmax().item()
            prob_pred = probs[pred].item()

        # Grad-CAM para la clase verdadera
        heatmap = cam(input_tensor=image_tensor,
                      targets=[ClassifierOutputTarget(cls_label)])[0]

        # Overlay
        img_np = norm(sample['image'].squeeze().cpu().numpy())
        img_rgb = np.stack([img_np]*3, axis=-1)
        cmap   = plt.get_cmap('jet')
        heat_rgb = cmap(heatmap)[..., :3]
        overlay = (0.55 * img_rgb + 0.45 * heat_rgb).clip(0, 1)

        # Imagen original (izquierda)
        ax_orig = axes[cls_label][col_offset * 2]
        ax_orig.imshow(img_np, cmap='gray')
        ax_orig.set_title(f'{img_name}\nTrue={CLASS_NAMES[cls_label]}',
                          fontsize=7, color=colors_title[cls_label])
        ax_orig.axis('off')

        # Grad-CAM (derecha)
        ax_cam = axes[cls_label][col_offset * 2 + 1]
        ax_cam.imshow(overlay)
        correct = '✓' if pred == cls_label else '✗'
        ax_cam.set_title(
            f'Grad-CAM | pred={CLASS_NAMES[pred]} (p={prob_pred:.2f}) {correct}',
            fontsize=7,
            color='green' if pred == cls_label else 'red'
        )
        ax_cam.axis('off')

    # Label de fila
    axes[cls_label][0].set_ylabel(CLASS_NAMES[cls_label],
                                   fontsize=11, fontweight='bold',
                                   color=colors_title[cls_label])

plt.suptitle('Grad-CAM — Sprint 2 (4 clases) — Test Set',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s2/figures/gradcam_s2.png',
            dpi=110, bbox_inches='tight')
plt.show()
print('✓ Grad-CAM guardado.')


## 8. Resumen final del Sprint 2

In [ ]:
print('=' * 65)
print('SPRINT 2 — BASELINE CNN 4 CLASES — CERRADO')
print('=' * 65)
print()
print('Tarea:   Clasificación multi-clase (4 clases)')
print('Dataset: NIH ChestX-ray14 (5 tarballs, ~25k imágenes disponibles)')
print('         739 train / 148 val / 148 test (1 img/paciente)')
print('Modelo:  DenseNet121 preentrenado, finetune FULL (6.95M params)')
print()
print('MÉTRICAS FINALES (sobre TEST, 148 imágenes nunca vistas):')
print(f"  AUC macro:    {m['auc_macro']:.4f}")
print(f"  Accuracy:     {m['accuracy']:.4f}")
print()
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<15}: AUC={m['auc_per_class'][i]:.4f} | "
          f"Sens={m['sensitivity_per_class'][i]:.4f} | "
          f"Spec={m['specificity_per_class'][i]:.4f}")
print()
print('COMPARATIVA S1 vs S2:')
print('  Sprint 1: 2 clases | 199 imgs train | AUC binario = 0.9121 (test)')
print(f"  Sprint 2: 4 clases | 739 imgs train | AUC macro   = {m['auc_macro']:.4f} (test)")
print()
print('Artefactos guardados en experiments_s2/:')
print('  - sprint2_best.pt                (checkpoint, epoch 40)')
print('  - history_s2.json                (curvas de entrenamiento)')
print('  - test_metrics_s2.json           (métricas finales)')
print('  - figures/curves_s2_40ep.png     (curvas de entrenamiento)')
print('  - figures/test_roc_cm_s2.png     (ROC + matriz de confusión)')
print('  - figures/gradcam_s2.png         (Grad-CAM 4 clases)')
print()
print('=' * 65)
print('SPRINT 2 CERRADO ✓')
print('=' * 65)
print()
print('Siguiente: Sprint 4 — Modelo híbrido CNN-ViT')
